# Data Inspection

This notebook inspects the raw datasets for the Recovery Access Gap Index project and creates the first cleaned municipality and overdose burden outputs.

## Project Setup


In [ ]:
from pathlib import Path

# Notebook is inside /notebooks, so parent folder is the project root
BASE_DIR = Path.cwd().parent
DATA_RAW = BASE_DIR / "data" / "raw"

print("Base directory:", BASE_DIR)
print("Raw data directory exists:", DATA_RAW.exists())

for folder in DATA_RAW.iterdir():
    print(folder.name)


## Inspect MassGIS municipality boundaries


In [ ]:
import geopandas as gpd

massgis_dir = DATA_RAW / "massgis_municipalities"

for shp in massgis_dir.rglob("*.shp"):
    print(shp.name)


In [ ]:
poly_path = massgis_dir / "TOWNSSURVEY_POLY.shp"

towns_poly = gpd.read_file(poly_path)

print("Rows:", len(towns_poly))
print("Unique towns:", towns_poly["TOWN"].nunique())
print("Columns:")
print(towns_poly.columns)

towns_poly.head()


In [ ]:
towns_muni = towns_poly.dissolve(
    by=["TOWN", "TOWN_ID", "COUNTY"],
    as_index=False
)

print("Rows after dissolve:", len(towns_muni))

towns_muni[["TOWN", "TOWN_ID", "COUNTY", "AREA_SQMI"]].head()


In [ ]:
towns_muni.plot()


In [ ]:
processed_dir = BASE_DIR / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "ma_municipalities_clean.geojson"

towns_muni.to_file(output_path, driver="GeoJSON")

print("Saved to:", output_path)
print(output_path.exists())


## Inspect Overdose Deaths


In [ ]:
from docx import Document

overdose_dir = DATA_RAW / "overdose_deaths"

print("Files in overdose deaths folder:")
for file in overdose_dir.iterdir():
    print(file.name)

In [ ]:
overdose_file = overdose_dir / "Opioid-related Overdose deaths by City or Town - June 2024.docx"

doc = Document(overdose_file)

print("Number of tables:", len(doc.tables))

for i, table in enumerate(doc.tables):
    print(f"\nTable {i}")
    print("Rows:", len(table.rows))
    print("Columns:", len(table.columns))
    
    # Print first 3 rows from each table
    for row in table.rows[:3]:
        cells = [cell.text.strip().replace("\n", " ") for cell in row.cells]
        print(cells)

In [ ]:
import pandas as pd

residence_tables = []

for i, table in enumerate(doc.tables):
    first_cell = table.rows[0].cells[0].text.strip()
    
    if "City/Town of Residence" in first_cell:
        rows = []
        for row in table.rows[1:]:  # skip first repeated header row
            cells = [cell.text.strip().replace("\n", " ") for cell in row.cells]
            rows.append(cells)
        
        df = pd.DataFrame(rows)
        df.columns = df.iloc[0]   # use second header row as columns
        df = df.iloc[1:]          # remove header row from data
        
        df["source_table"] = i
        residence_tables.append(df)

overdose_residence = pd.concat(residence_tables, ignore_index=True)

print("Rows:", len(overdose_residence))
print("Columns:")
print(overdose_residence.columns)

overdose_residence.head()

In [ ]:
year_cols = ["2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023"]

overdose_clean = overdose_residence.copy()

# Convert year columns to numeric
for col in year_cols:
    overdose_clean[col] = pd.to_numeric(overdose_clean[col], errors="coerce")

# Create project metric: 2021–2023 average annual overdose deaths
overdose_clean["avg_deaths_2021_2023"] = overdose_clean[["2021", "2022", "2023"]].mean(axis=1)

# Standardized town name for joining later
overdose_clean["town_join"] = overdose_clean["City/Town of Residence"].str.upper().str.strip()

overdose_clean[
    ["City/Town of Residence", "2021", "2022", "2023", "avg_deaths_2021_2023", "town_join"]
].head(20)

In [ ]:
overdose_clean.sort_values("avg_deaths_2021_2023", ascending=False)[
    ["City/Town of Residence", "2021", "2022", "2023", "avg_deaths_2021_2023"]
].head(20)

In [ ]:
output_path = processed_dir / "overdose_deaths_by_town_clean.csv"

overdose_clean.to_csv(output_path, index=False)

print("Saved to:", output_path)
print(output_path.exists())

## Inspect EMS Incidents

In [ ]:
ems_dir = DATA_RAW / "ems_incidents"

print("Files in EMS folder:")
for file in ems_dir.iterdir():
    print(file.name)

In [ ]:
ems_file = ems_dir / "Emergency Medical Services Data - June 2024.docx"

ems_doc = Document(ems_file)

print("Number of tables:", len(ems_doc.tables))

for i, table in enumerate(ems_doc.tables):
    print(f"\nTable {i}")
    print("Rows:", len(table.rows))
    print("Columns:", len(table.columns))
    
    for row in table.rows[:3]:
        cells = [cell.text.strip().replace("\n", " ") for cell in row.cells]
        print(cells)

In [ ]:
ems_tables = []

for i, table in enumerate(ems_doc.tables):
    first_cell = table.rows[0].cells[0].text.strip()
    
    if "All Suspected Opioid-Related Incidents by Town" in first_cell:
        rows = []
        
        # Skip title row and year row, use the third row as header
        for row in table.rows[2:]:
            cells = [cell.text.strip().replace("\n", " ") for cell in row.cells]
            rows.append(cells)
        
        df = pd.DataFrame(rows)
        df.columns = [
            "city_town",
            "2022_q1", "2022_q2", "2022_q3", "2022_q4", "2022_total",
            "2023_q1", "2023_q2", "2023_q3", "2023_q4", "2023_total"
        ]
        
        # Remove repeated header rows if any
        df = df[df["city_town"] != "City/Town"]
        
        df["source_table"] = i
        ems_tables.append(df)

ems_incidents = pd.concat(ems_tables, ignore_index=True)

print("Rows:", len(ems_incidents))
print("Columns:")
print(ems_incidents.columns)

ems_incidents.head()

In [ ]:
ems_clean = ems_incidents.copy()

def clean_ems_value(value):
    value = str(value).strip()
    
    if value in ["†", ""]:
        return 0
    
    if value in ["(1-4)", "*"]:
        return None
    
    value = value.replace(",", "").replace("±", "")
    
    try:
        return float(value)
    except ValueError:
        return None

for col in ["2022_total", "2023_total"]:
    ems_clean[col] = ems_clean[col].apply(clean_ems_value)

ems_clean["avg_ems_incidents_2022_2023"] = ems_clean[["2022_total", "2023_total"]].mean(axis=1)
ems_clean["town_join"] = ems_clean["city_town"].str.upper().str.strip()

ems_clean[["city_town", "2022_total", "2023_total", "avg_ems_incidents_2022_2023", "town_join"]].head(20)

In [ ]:
ems_clean.sort_values("avg_ems_incidents_2022_2023", ascending=False)[
    ["city_town", "2022_total", "2023_total", "avg_ems_incidents_2022_2023"]
].head(20)

In [ ]:
output_path = processed_dir / "ems_incidents_by_town_clean.csv"

ems_clean.to_csv(output_path, index=False)

print("Saved to:", output_path)
print(output_path.exists())